# Campaign — every `results/` directory in one go

The six configurations the direction reports, in one kernel and one cell: no restarting between
`VARIANT`s, no rebuilding the feature store between `SPLIT`s, no run that exists only in a Colab
session. The loop is [`training.py`](training.py); this notebook is the settings plus one call.

What it writes, per run: `results/<variant>-<split>/` with `run.json`, `summary.json`,
`folds.csv`, `preds.csv`, `board_row.md` — the layout is owned by
[`results/README.md`](results/README.md). Six runs × 3 seeds = **72 trainings**: 15 per run on the
`subject` split (5 folds), 9 on the `official` one (3 folds). The `subject` folds stop early on
validation AUC; the `official` ones run the full 30 epochs, because that split has no validation
set to stop on.

Order matters only for time: the splits are grouped, so the 330 MB cache is read once per
protocol instead of once per run.

## 0 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# The repo is needed for two things that are NOT on Drive: the `lib/` code and the
# frozen splits (they are versioned in git, that is what makes them frozen).
import os
if not os.path.isdir("/content/DeepLearningCourse/.git"):
    !git clone -q https://github.com/mattsays/DeepLearningCourse /content/DeepLearningCourse
%cd /content/DeepLearningCourse/lie-detector
!git pull -q
import sys; sys.path.insert(0, '.')

## 1 — Settings

`RUNS` is the whole campaign: a list of `(variant, split)` pairs. `variant` is a key of
`model.VARIANTS`; `split` is `"subject"` (our protocol, no speaker crosses the boundary) or
`"official"` (the paper's clip-level 3-fold, which leaks speaker and round — reproduced to
measure the leak, never to compete). `congruence` is deliberately not in the list: it is a
diagnostic probe, not a candidate for the board.

`SEEDS` is three for every row, including the controls: a one-seed row cannot be told apart from
a lucky draw, and the seed spread on this dataset is larger than the differences between
variants.

In [ ]:
from pathlib import Path

# The one shared folder. Same path for everybody: add a Drive shortcut named
# "dolos" in your My Drive and this resolves without touching anything else.
DATA = Path("/content/drive/MyDrive/dolos")

RUNS = [
    ("late", "subject"),      # the board row: pool + concat + MLP
    ("audio", "subject"),     # control: does the fused model beat WavLM alone?
    ("video", "subject"),     # control: does the face carry anything at 16 frames?
    ("late", "official"),     # the same three on the leaky split, for the protocol effect
    ("audio", "official"),
    ("video", "official"),
]

SEEDS = [0, 1, 2]             # every row, controls included
EPOCHS = 30                   # the official split runs all of them: it has no val to stop on
PATIENCE = 8                  # subject only: early stopping on validation AUC
BATCH_SIZE = 16
PROTOCOL = "v1"               # "v1" board protocol, "v2" purity-balanced ablation
CAPTAINS = "include"          # "exclude" drops the 468 clips of the two recurring captains
READ_WORKERS = 64             # parallel readers for the Drive mount: it is latency-bound

import torch
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print("device:", DEVICE)
print(DATA, "->", sorted(p.name for p in DATA.glob("*")))

## 2 — The direction's own code

`training.py` and `model.py` live beside this notebook, which is not the folder the setup cell
`%cd`s into: put it on the path once, here.

In [ ]:
import sys

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "model.py").exists():
    NOTEBOOK_DIR = Path.cwd() / "notebooks/architectures/cross-modal-fusion"
sys.path.insert(0, str(NOTEBOOK_DIR))
print("notebook dir:", NOTEBOOK_DIR)

## 3 — Protocol, clips, and a pre-flight check on the caches

`Protocol.load` verifies the manifest hash: a hand-edited split raises instead of silently
scoring a different protocol. `Clips.describe_source()` is the fingerprint to post in the group
chat — two people with different hashes are not training on the same data.

The cache check is there so a missing encoder fails in two seconds rather than after the first
of ninety trainings.

In [ ]:
from lib.dataset import Clips, Protocol, cache_dir

from training import Encoders, Settings

settings = Settings(data=DATA, encoders=Encoders(), seeds=list(SEEDS), epochs=EPOCHS,
                    patience=PATIENCE, batch_size=BATCH_SIZE, read_workers=READ_WORKERS,
                    protocol=PROTOCOL, captains=CAPTAINS, device=DEVICE)

missing = [name for name in (settings.encoders.video, settings.encoders.audio,
                             settings.encoders.text)
           if not cache_dir(name, DATA).exists()]
if missing:
    raise SystemExit(f"missing feature caches: {missing} — run extract_features.ipynb first")

folds = Protocol.load(settings.splits())
clips = Clips.load(DATA)
print(clips.describe_source())
print(f"{folds.folds} folds, protocol {PROTOCOL}, captains {CAPTAINS}")

## 4 — The campaign

One call. It prints a line per fold and a summary table, and writes `results/<run>/` after each
run, so a crash in run 5 leaves runs 1–4 on disk.

When it finishes, the folder to hand over is `results/`: `git add results && git commit` from
Colab, or zip it.

In [ ]:
from training import run_campaign

board = run_campaign(clips, folds, settings, runs=RUNS, root=NOTEBOOK_DIR / "results")